# analysis_scene_2 업데이트
**새 프롬프트로 analysis_scene_2의 context_narrative를 업데이트합니다.**

실행 전 체크리스트:
- [ ] 런타임 > 런타임 유형 변경 > **GPU (T4)** 선택
- [ ] Google Drive에 `hyuck1.mp4` 업로드 확인
  - 경로: `내 드라이브/SampleVideo.Test/hyuck1.mp4`

## 0. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 패키지 설치
!pip install -q transformers accelerate qwen-vl-utils psycopg2-binary
print("설치 완료!")

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
VIDEO_PATH = "/content/drive/MyDrive/SampleVideo.Test/hyuck1.mp4"
assert os.path.exists(VIDEO_PATH), f"영상 없음: {VIDEO_PATH}"
print(f"영상 확인 완료: {VIDEO_PATH}")

## 1. Qwen2-VL 모델 로드

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

print("모델 로딩 중...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("모델 로드 완료!")

## 2. 프롬프트 정의

In [ ]:
NEW_PROMPT = (
    "이 영상 씬을 분석하여 아래 세 항목을 각각 정확히 1문장으로 작성하세요.\n"
    "반드시 'label: 내용' 형식을 지켜야 합니다.\n\n"
    "상황: 이 씬에 등장하는 장면, 배경, 인물의 행동을 묘사하세요. (감정·느낌 표현 금지)\n"
    "감정: 이 씬이 시청자에게 전달하는 감성적 분위기나 정서를 표현하세요. (상황 묘사 금지)\n"
    "욕구: 이 씬이 타겟하는 시청자의 내면적 니즈나 욕구를 서술하세요. (감정 단어·상황 묘사 금지)\n\n"
    "예시:\n"
    "상황: 퇴근 후 집에 돌아온 직장인이 소파에 앉아 따뜻한 음료를 마시는 장면이다.\n"
    "감정: 하루의 피로가 녹아드는 포근하고 안도감 있는 분위기를 전달한다.\n"
    "욕구: 바쁜 일상 속에서 잠깐의 휴식과 자신을 위한 작은 여유를 원하는 사람에게 어울린다.\n\n"
    "반드시 한국어로만 작성하고, 위 예시처럼 세 줄로만 답하세요."
)
print("프롬프트 설정 완료!")

## 3. 전체 씬 분석 및 DB 업데이트

In [ ]:
import cv2
import psycopg2
from tqdm.auto import tqdm

# DB 연결 정보
DB_CONFIG = {
    "host":     "121.167.223.17",
    "port":     5432,
    "dbname":   "hv02",
    "user":     "postgres01",
    "password": "postgres01"
}

# DB에서 씬 목록 가져오기
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()
cur.execute("SELECT id, scene_start_sec, scene_end_sec FROM public.analysis_scene_2 ORDER BY id")
scenes = cur.fetchall()
print(f"총 씬 수: {len(scenes)}")

# 영상 로드
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"영상 FPS: {fps:.1f}")

fail_count = 0

for scene_id, start_sec, end_sec in tqdm(scenes, desc="씬 분석 중"):
    # 씬 중간 프레임 추출
    mid_sec = (start_sec + end_sec) / 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(mid_sec * fps))
    ret, frame = cap.read()

    if not ret:
        narrative = "프레임 추출 실패"
        fail_count += 1
    else:
        frame_path = "/tmp/frame_tmp.jpg"
        cv2.imwrite(frame_path, frame)

        # 추론
        messages = [{"role": "user", "content": [
            {"type": "image", "image": frame_path},
            {"type": "text",  "text": NEW_PROMPT},
        ]}]
        text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[frame_path], padding=True, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=200,
                repetition_penalty=1.2,
                do_sample=False,
            )
        trimmed   = generated[0][inputs['input_ids'].shape[-1]:]
        narrative = processor.decode(trimmed, skip_special_tokens=True).strip()

    # DB 업데이트
    cur.execute(
        "UPDATE public.analysis_scene_2 SET context_narrative = %s WHERE id = %s",
        (narrative, scene_id)
    )
    conn.commit()

cap.release()
cur.close()
conn.close()

print(f"\n모든 씬 업데이트 완료! (실패: {fail_count}건)")

## 4. 결과 확인 (샘플 5개)

In [ ]:
import psycopg2
from IPython.display import display, HTML

conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

cur.execute("""
    SELECT
        a1.id,
        a1.scene_start_sec,
        a1.scene_end_sec,
        a1.context_narrative AS 기존_narrative,
        a2.context_narrative AS 신규_narrative
    FROM public.analysis_scene   a1
    JOIN public.analysis_scene_2 a2 ON a1.id = a2.id
    ORDER BY a1.id
    LIMIT 5
""")
rows = cur.fetchall()
cur.close()
conn.close()

html = "<h3>기존 vs 신규 프롬프트 결과 비교 (5건)</h3>"
for row in rows:
    scene_id, start, end, old_txt, new_txt = row
    old_txt = (old_txt or "")[:300]
    new_txt = (new_txt or "")[:300]
    html += f"""
    <div style="border:1px solid #ccc; border-radius:8px; padding:12px; margin:8px 0;">
      <b>ID {scene_id} | {start:.1f}s ~ {end:.1f}s</b>
      <div style="display:flex; gap:12px; margin-top:8px;">
        <div style="flex:1; background:#FFE0E0; border-radius:6px; padding:10px;">
          <b style="color:red;">[기존]</b>
          <p style="font-size:13px; white-space:pre-wrap; margin-top:6px;">{old_txt}</p>
        </div>
        <div style="flex:1; background:#E0FFE0; border-radius:6px; padding:10px;">
          <b style="color:green;">[신규]</b>
          <p style="font-size:13px; white-space:pre-wrap; margin-top:6px;">{new_txt}</p>
        </div>
      </div>
    </div>
    """

display(HTML(html))